In [4]:
import logging
import warnings
import os
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from datetime import datetime
import hashlib

from dotenv import load_dotenv
from bs4 import BeautifulSoup
import openai
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import re
import time

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

load_dotenv("../.env", override=True)

True

## 1. Konfigurering og dataklassr

In [5]:
# Konfigurering
@dataclass
class EmbeddingConfig:
    """Konfigurering for embedding pipeline"""
    chunk_size: int = 1200  # Tokens per chunk (increased for semantic chunks)
    chunk_overlap: int = 100  # Less overlap for semantic boundaries
    embedding_model: str = "text-embedding-3-large"  # OpenAI embedding model
    elasticsearch_index: str = "lovdata_semantic_ada3l_251108"  # Production index
    batch_size: int = 50  # Batch size for embedding/ingest
    
@dataclass 
class DocumentChunk:
    """Representerer en chunk av et dokument - kompatibel med eksisterende app"""
    text: str
    metadata: Dict[str, Any]
    id_: str = ""

@dataclass
class LawMetadata:
    """Metadata for legal documents"""
    document_id: str  # LOV-1987-06-12-48
    title: str
    department: str
    legal_areas: List[str]
    last_modified: str
    short_title: str = ""
    ref_id: str = ""

@dataclass
class LawArticle:
    """Individual legal article/section"""
    article_number: str  # §1, §2, etc.
    title: str
    content: str
    section_title: str
    hierarchy_path: str
    token_count: int = 0
    chunk_type: str = "article"

# Initialize config
config = EmbeddingConfig()
openai.api_key = os.environ.get("OPENAI_API_KEY")

# ES connection exactly like the working notebook
es_client = Elasticsearch(
    ["https://elasticsearch-llm-spring-glitter-3589.fly.dev"],
    basic_auth=('elastic', os.environ.get("ELASTIC_PASSWORD")),
    verify_certs=False
)

print(f"Config: {config}")
print(f"ES health: {es_client.cluster.health()}")

Config: EmbeddingConfig(chunk_size=1200, chunk_overlap=100, embedding_model='text-embedding-3-large', elasticsearch_index='lovdata_semantic_ada3l_251108', batch_size=50)
ES health: {'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 7, 'active_shards': 7, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 6, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 53.84615384615385}


## 2. XML Parsing og Lovdata Processing

In [6]:
def parse_lovdata_xml(xml_path: Path) -> tuple[LawMetadata, List[LawArticle]]:
    """Parse Lovdata XML file into metadata and articles"""
    
    with open(xml_path, 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file, 'html.parser')
    
    # Extract metadata
    metadata = extract_metadata(soup)
    
    # Extract all articles
    articles = extract_articles(soup, metadata.document_id)
    
    return metadata, articles

def extract_metadata(soup: BeautifulSoup) -> LawMetadata:
    """Extract document metadata from header section"""
    
    def safe_get_text(element):
        return element.get_text().strip() if element else ""
    
    document_id = safe_get_text(soup.find('dd', class_='legacyID'))
    title = safe_get_text(soup.find('dd', class_='title'))
    department = safe_get_text(soup.find('dd', class_='ministry'))
    last_modified = safe_get_text(soup.find('dd', class_='lastChangeInForce'))
    short_title = safe_get_text(soup.find('dd', class_='titleShort'))
    ref_id = safe_get_text(soup.find('dd', class_='refid'))
    
    # Extract legal areas
    legal_areas = []
    legal_area_dd = soup.find('dd', class_='legalArea')
    if legal_area_dd:
        for link in legal_area_dd.find_all('a'):
            legal_areas.append(link.get_text().strip())
    
    return LawMetadata(
        document_id=document_id,
        title=title,
        department=department,
        legal_areas=legal_areas,
        last_modified=last_modified,
        short_title=short_title,
        ref_id=ref_id
    )

def estimate_token_count(text: str) -> int:
    """Estimate token count (4 characters per token for Norwegian)"""
    return len(text) // 4

def extract_articles(soup: BeautifulSoup, document_id: str) -> List[LawArticle]:
    """Extract all legal articles from the document"""
    
    articles = []
    
    # Find all legal articles
    for article_elem in soup.find_all('article', class_='legalArticle'):
        article = extract_single_article(article_elem, document_id)
        if article and article.content.strip():  # Only articles with content
            articles.append(article)
    
    return articles

def extract_single_article(article_elem, document_id: str) -> Optional[LawArticle]:
    """Extract a single article with its content"""
    
    # Get article number and title
    header = article_elem.find('h3', class_='legalArticleHeader')
    if not header:
        return None
        
    article_num_elem = header.find('span', class_='legalArticleValue')
    article_title_elem = header.find('span', class_='legalArticleTitle')
    
    article_number = article_num_elem.get_text().strip() if article_num_elem else ""
    article_title = article_title_elem.get_text().strip() if article_title_elem else ""
    
    # Find section context
    section = article_elem.find_parent('section')
    section_title = ""
    if section and section.find('h2'):
        section_title = section.find('h2').get_text().strip()
    
    # Extract all text content from article paragraphs
    content_parts = []
    for p in article_elem.find_all('article', class_='legalP'):
        text = p.get_text().strip()
        if text:
            content_parts.append(text)
    
    content = "\\n\\n".join(content_parts)
    
    # Create hierarchy path
    hierarchy_path = f"{document_id}/{article_number}"
    
    return LawArticle(
        article_number=article_number,
        title=article_title,
        content=content,
        section_title=section_title,
        hierarchy_path=hierarchy_path,
        token_count=estimate_token_count(content),
        chunk_type="article"
    )

def create_document_chunks_from_law(xml_path: Path, config: EmbeddingConfig) -> List[DocumentChunk]:
    """
    Create DocumentChunk objects from Lovdata XML - maintains app compatibility
    
    Args:
        xml_path: Path to Lovdata XML file
        config: EmbeddingConfig
        
    Returns:
        List of DocumentChunk objects compatible with existing app
    """
    metadata, articles = parse_lovdata_xml(xml_path)
    
    chunks = []
    document_name = metadata.document_id  # Use document_id as document_name
    
    for i, article in enumerate(articles):
        # Create rich metadata that preserves all Lovdata structure
        chunk_metadata = {
            'source_file': xml_path.name,
            'created_at': datetime.now().isoformat(),
            'estimated_tokens': article.token_count,
            
            # Lovdata-specific metadata (new)
            'document_id': metadata.document_id,
            'document_title': metadata.title,
            'article_number': article.article_number,
            'article_title': article.title,
            'section_title': article.section_title,
            'hierarchy_path': article.hierarchy_path,
            'chunk_type': article.chunk_type,
            'legal_areas': metadata.legal_areas,
            'department': metadata.department,
            'last_modified': metadata.last_modified,
            'ref_id': metadata.ref_id
        }
        
        chunk = DocumentChunk(
            text=f"{article.title}\\n\\n{article.content}",  # Include title in content
            document_name=document_name,
            chunk_index=i,
            total_chunks=len(articles),
            metadata=chunk_metadata
        )
        chunks.append(chunk)
    
    return chunks

## 3. Semantic Chunking with Size Management

In [7]:
def chunk_oversized_article(article: LawArticle, max_tokens: int = 1200, overlap_tokens: int = 100) -> List[LawArticle]:
    """
    Split oversized articles into smaller chunks while preserving context
    """
    if article.token_count <= max_tokens:
        return [article]
    
    # Split content into paragraphs
    paragraphs = [p.strip() for p in article.content.split('\\n\\n') if p.strip()]
    
    chunks = []
    current_content = ""
    current_tokens = 0
    chunk_num = 0
    
    for paragraph in paragraphs:
        para_tokens = estimate_token_count(paragraph)
        
        # If adding this paragraph exceeds max_tokens, create a chunk
        if current_tokens + para_tokens > max_tokens and current_content:
            chunks.append(create_sub_article(article, current_content, chunk_num))
            
            # Start new chunk with overlap (use last sentence if available)
            if overlap_tokens > 0 and len(current_content) > overlap_tokens * 4:
                overlap_text = current_content[-(overlap_tokens * 4):]
                # Try to start at sentence boundary
                sentence_start = overlap_text.find('. ')
                if sentence_start != -1:
                    overlap_text = overlap_text[sentence_start + 2:]
                current_content = overlap_text + "\\n\\n" + paragraph
            else:
                current_content = paragraph
                
            current_tokens = estimate_token_count(current_content)
            chunk_num += 1
        else:
            # Add paragraph to current chunk
            if current_content:
                current_content += "\\n\\n" + paragraph
            else:
                current_content = paragraph
            current_tokens += para_tokens
    
    # Add the last chunk if there's content
    if current_content:
        chunks.append(create_sub_article(article, current_content, chunk_num))
    
    return chunks if chunks else [article]  # Return original if splitting failed

def create_sub_article(original_article: LawArticle, content: str, chunk_num: int) -> LawArticle:
    """Create a sub-article from oversized article"""
    return LawArticle(
        article_number=f"{original_article.article_number}-{chunk_num + 1}",
        title=original_article.title,
        content=content,
        section_title=original_article.section_title,
        hierarchy_path=f"{original_article.hierarchy_path}-{chunk_num + 1}",
        token_count=estimate_token_count(content),
        chunk_type="article_part"
    )

def process_xml_files_semantically(xml_files: List[Path], config: EmbeddingConfig) -> List[DocumentChunk]:
    """
    Process multiple XML files with semantic chunking
    """
    all_chunks = []
    
    for xml_file in xml_files:
        print(f"📄 Processing: {xml_file.stem}")
        
        try:
            metadata, articles = parse_lovdata_xml(xml_file)
            
            # Process each article, splitting if needed
            processed_articles = []
            oversized_count = 0
            
            for article in articles:
                if article.token_count > config.chunk_size:
                    # Split oversized articles
                    sub_articles = chunk_oversized_article(article, config.chunk_size, config.chunk_overlap)
                    processed_articles.extend(sub_articles)
                    oversized_count += 1
                    print(f"  📝 Split {article.article_number} ({article.token_count} tokens) into {len(sub_articles)} parts")
                else:
                    processed_articles.append(article)
            
            # Convert to DocumentChunks - FIXED VERSION
            for i, article in enumerate(processed_articles):
                chunk_metadata = {
                    'source_file': xml_file.name,
                    'created_at': datetime.now().isoformat(),
                    'estimated_tokens': article.token_count,
                    'document_id': metadata.document_id,
                    'document_title': metadata.title,
                    'article_number': article.article_number,
                    'article_title': article.title,
                    'section_title': article.section_title,
                    'hierarchy_path': article.hierarchy_path,
                    'chunk_type': article.chunk_type,
                    'legal_areas': metadata.legal_areas,
                    'department': metadata.department,
                    'last_modified': metadata.last_modified,
                    'ref_id': metadata.ref_id
                }
                
                # FIXED: Use only the parameters that DocumentChunk accepts
                chunk = DocumentChunk(
                    text=f"{article.title}\\n\\n{article.content}",
                    metadata=chunk_metadata
                )
                all_chunks.append(chunk)
            
            print(f"  ✅ {len(articles)} articles → {len(processed_articles)} chunks ({oversized_count} split)")
            
        except Exception as e:
            print(f"  ❌ Error processing {xml_file.stem}: {e}")
    
    return all_chunks

## 4. Enhanced ES Index and OpenAI Integration

In [8]:
def embed_chunks(chunks: List[DocumentChunk]) -> List[DocumentChunk]:
    """Embed chunks with OpenAI"""
    embedded_chunks = []
    for i, chunk in enumerate(chunks):
        response = openai.embeddings.create(
            input=chunk.text,
            model=config.embedding_model
        )
        
        chunk.metadata['embedding'] = response.data[0].embedding
        chunk.metadata['embedding_model'] = config.embedding_model
        embedded_chunks.append(chunk)
        
    return embedded_chunks

def ingest_to_elasticsearch(chunks: List[DocumentChunk], index_name: str) -> int:
    """Simple ingest to Elasticsearch - exactly like the working notebook"""
    count = 0
    for chunk in chunks:
        if 'embedding' in chunk.metadata:
            doc = {
                "content": chunk.text,
                "document_id": chunk.metadata.get('document_id', ''),
                "article_number": chunk.metadata.get('article_number', ''),
                "embedding": chunk.metadata['embedding']
            }
            
            es_client.index(index=index_name, body=doc)
            count += 1
    
    return count

In [9]:
def is_tax_relevant_document(metadata: LawMetadata, articles: List[LawArticle]) -> bool:
    """
    Filter documents for tax relevance - REFINED approach with stricter keywords
    
    BALANCED LOGIC:
    1. Include ALL documents from core tax departments (no keyword filtering)
    2. Apply STRICT keyword filtering to other departments
    """
    
    # REFINED tax keywords - more specific, removed generic terms
    tax_keywords = [
        # Core tax terms
        'skatt', 'skatteloven', 'skatteetaten', 'skatteoppkrever', 
        'inntektsskatt', 'formuesskatt', 'selskapsskatt', 'eiendomsskatt',
        
        # Specific tax types
        'merverdiavgift', 'mva', 'toll', 'tollavgift',
        'særavgift', 'arveavgift', 'dokumentavgift', 'stempelavgift',
        
        # Tax-specific processes
        'skatteberegning', 'avgiftssats', 'avgiftsfri', 'skattefri', 
        'avgiftsplikt', 'skatteplikt',
        
        # Removed: 'gebyr', 'avgift' (too generic - catches education, admin fees)
    ]
    
    # Core tax departments - include ALL their documents 
    core_tax_departments = [
        'Finansdepartementet',              # Primary tax department
        'Nærings- og fiskeridepartementet', # Business regulations, VAT, industry taxes
    ]
    
    # Semi-relevant departments - apply STRICT keyword filtering (REDUCED LIST)
    keyword_filtered_departments = [
        'Justis- og beredskapsdepartementet', # Legal framework, enforcement
        'Arbeids- og sosialdepartementet',  # Payroll taxes, social security
        'Samferdselsdepartementet',         # Transport taxes, vehicle taxes, fuel taxes
        'Klima- og miljødepartementet',     # Environmental taxes, CO2 taxes
        'Landbruks- og matdepartementet',   # Agricultural taxes, land taxes
    ]
    
    # Check if document is from core tax departments
    from_core_dept = any(dept in metadata.department for dept in core_tax_departments)
    
    if from_core_dept:
        print(f"  ✅ Tax relevant (core dept): {metadata.document_id} - {metadata.department}")
        return True
    
    # For semi-relevant and other departments, check for STRICT tax keywords
    title_lower = metadata.title.lower()
    title_match = any(keyword in title_lower for keyword in tax_keywords)
    
    if title_match:
        dept_type = "semi-relevant" if any(dept in metadata.department for dept in keyword_filtered_departments) else "other"
        print(f"  ✅ Tax relevant (title keywords - {dept_type}): {metadata.document_id}")
        print(f"     Title: {metadata.title[:80]}...")
        return True
    
    # Check content for STRICT tax keywords (sample first few articles)
    content_match = False
    sample_articles = articles[:3]  # Check first 3 articles
    for article in sample_articles:
        content_lower = article.content.lower()
        if any(keyword in content_lower for keyword in tax_keywords):
            content_match = True
            break
    
    if content_match:
        dept_type = "semi-relevant" if any(dept in metadata.department for dept in keyword_filtered_departments) else "other"
        print(f"  ✅ Tax relevant (content keywords - {dept_type}): {metadata.document_id}")
        print(f"     Department: {metadata.department}")
        return True
    
    return False


def filter_xml_files_for_tax(xml_files: List[Path], max_to_check: int = None) -> List[Path]:
    """
    Filter XML files for tax relevance - REFINED approach
    
    STRICT STRATEGY:
    - ALL files from 2 core tax departments  
    - STRICT keyword-filtered files from 5 semi-relevant departments (REDUCED)
    - STRICT keyword-filtered files from all other departments
    """
    if max_to_check:
        xml_files = xml_files[:max_to_check]
        
    print(f"🎯 Filtering {len(xml_files)} files for tax relevance...")
    print("📋 REFINED FILTERING STRATEGY (REDUCED):")
    print("   ✅ ALL docs from:")
    print("     🏦 Finansdepartementet")
    print("     🏢 Nærings- og fiskeridepartementet") 
    print("   🔍 STRICT KEYWORD-FILTERED docs from:")
    print("     ⚖️ Justis- og beredskapsdepartementet")
    print("     👥 Arbeids- og sosialdepartementet")
    print("     🚗 Samferdselsdepartementet")
    print("     🌱 Klima- og miljødepartementet")
    print("     🌾 Landbruks- og matdepartementet")
    print("     📋 All other departments")
    print("   🚫 REMOVED DEPTS: Kommunal/inkludering (municipal focus)")
    print("   🚫 REMOVED KEYWORDS: 'gebyr', 'avgift' (too generic)")
    
    tax_relevant_files = []
    core_included = []
    keyword_included = []
    
    # Track departments we encounter
    departments_found = set()
    
    for xml_file in xml_files:
        try:
            metadata, articles = parse_lovdata_xml(xml_file)
            departments_found.add(metadata.department)
            
            if is_tax_relevant_document(metadata, articles):
                tax_relevant_files.append(xml_file)
                
                # Categorize inclusion reason for reporting
                core_departments = ['Finansdepartementet', 'Nærings- og fiskeridepartementet']
                
                if any(dept in metadata.department for dept in core_departments):
                    core_included.append(xml_file)
                else:
                    keyword_included.append(xml_file)
                
        except Exception as e:
            print(f"  ❌ Error checking {xml_file.stem}: {e}")
    
    reduction_pct = (1 - len(tax_relevant_files) / len(xml_files)) * 100
    core_pct = (len(core_included) / len(tax_relevant_files)) * 100 if tax_relevant_files else 0
    keyword_pct = (len(keyword_included) / len(tax_relevant_files)) * 100 if tax_relevant_files else 0
    
    print(f"\n📊 REFINED FILTERING RESULTS:")
    print(f"  📄 Original files: {len(xml_files)}")
    print(f"  ✅ Tax relevant total: {len(tax_relevant_files)}")
    print(f"     🏦 From core departments: {len(core_included)} ({core_pct:.1f}%)")
    print(f"     🔍 From keyword filtering: {len(keyword_included)} ({keyword_pct:.1f}%)")
    print(f"  📉 Reduction: {reduction_pct:.1f}%")
    
    return tax_relevant_files

In [10]:
# 🚀 PRODUCTION PROCESSING - Run this to ingest the filtered dataset

def run_production_ingestion():
    """
    Process the full Lovdata dataset with refined filtering
    """
    data_dir = Path("../data/extracted")
    
    print("🚀 STARTING PRODUCTION INGESTION")
    print("=" * 60)
    print(f"📊 Target index: {config.elasticsearch_index}")
    print(f"📂 Data directory: {data_dir}")
    print(f"🔍 Filtering strategy: 2 core depts + 5 keyword-filtered + strict keywords")
    print(f"⚡ Processing in batches of {config.batch_size}")
    
    # Run the full processing with refined filtering
    results = process_lovdata_dataset_production(
        data_dir=data_dir,
        config=config,
        max_files=None,  # Process all files
        use_tax_filter=True  # Apply refined filtering
    )
    
    # Print final summary
    print("\n" + "="*60)
    print("🏁 PRODUCTION INGESTION COMPLETE")
    print("="*60)
    print(f"📄 Total XML files found: {results.get('files_before_filter', 'N/A')}")
    print(f"✅ Tax-relevant files processed: {results['processed_files']}")
    print(f"📉 Document reduction: {results.get('reduction_pct', 0):.1f}%")
    print(f"📝 Total legal articles: {results.get('total_articles', 0)}")
    print(f"📦 Total chunks created: {results['total_chunks']}")
    print(f"🤖 Chunks embedded: {results['total_embedded']}")
    print(f"📇 Chunks ingested to ES: {results['total_ingested']}")
    print(f"⏱️ Total processing time: {results['processing_time'] / 3600:.1f} hours")
    print(f"📈 Processing rate: {results.get('files_per_hour', 0):.1f} files/hour")
    
    if results['total_ingested'] > 0:
        print(f"\n✅ SUCCESS: Index '{config.elasticsearch_index}' ready for production!")
        print(f"💡 You can now update your app to use this index")
        print(f"🔄 To update app: Change ELASTICSEARCH_INDEX_SKATT to '{config.elasticsearch_index}'")
    else:
        print(f"\n❌ ERROR: No documents were ingested!")
    
    return results

def process_lovdata_dataset_production(data_dir: Path, config: EmbeddingConfig, max_files: int = None, use_tax_filter: bool = True) -> Dict[str, Any]:
    """
    Production version with clean progress reporting
    """
    import time
    
    # Find all XML files
    xml_files = list(data_dir.glob("**/*.xml"))
    original_file_count = len(xml_files)
    
    if max_files:
        xml_files = xml_files[:max_files]
    
    print(f"📊 Found {len(xml_files)} XML files in dataset")
    
    # Apply tax filtering with minimal output
    if use_tax_filter:
        print(f"🔍 Applying refined tax filtering...")
        xml_files = filter_xml_files_for_tax_production(xml_files, max_to_check=max_files)
        reduction_pct = (1 - len(xml_files) / original_file_count) * 100
        print(f"✅ Filtering complete: {len(xml_files)} tax-relevant files ({reduction_pct:.1f}% reduction)")
    else:
        reduction_pct = 0
        print(f"📊 Processing all {len(xml_files)} files (no filtering)")
    
    results = {
        "processed_files": 0,
        "total_articles": 0,
        "total_chunks": 0,
        "total_embedded": 0,
        "total_ingested": 0,
        "processing_time": 0,
        "files_before_filter": original_file_count,
        "files_after_filter": len(xml_files),
        "reduction_pct": reduction_pct,
        "files_per_hour": 0
    }
    
    if not xml_files:
        print("❌ No files to process after filtering!")
        return results
        
    start_time = time.time()
    
    # Process in batches with clean progress
    batch_size = config.batch_size
    total_batches = (len(xml_files) + batch_size - 1) // batch_size
    
    print(f"⚡ Processing {len(xml_files)} files in {total_batches} batches")
    print(f"📈 Progress: [", end="", flush=True)
    
    for batch_num in range(total_batches):
        batch_start = time.time()
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(xml_files))
        batch_files = xml_files[start_idx:end_idx]
        
        # Process semantic chunks (silent)
        batch_chunks = process_xml_files_semantically_silent(batch_files, config)
        
        if batch_chunks:
            # Embed chunks (silent)
            embedded_chunks = embed_chunks_silent(batch_chunks)
            
            # Ingest to Elasticsearch (silent)
            ingested_count = ingest_to_elasticsearch_silent(embedded_chunks, config.elasticsearch_index)
            
            results["total_embedded"] += len(embedded_chunks)
            results["total_ingested"] += ingested_count
            results["total_chunks"] += len(batch_chunks)
        
        results["processed_files"] += len(batch_files)
        
        # Progress indicator
        progress_char = "█" if (batch_num + 1) % max(1, total_batches // 20) == 0 else "▓"
        print(progress_char, end="", flush=True)
        
        # Time estimation every 10% 
        if (batch_num + 1) % max(1, total_batches // 10) == 0:
            elapsed = time.time() - start_time
            remaining_batches = total_batches - batch_num - 1
            if remaining_batches > 0:
                avg_batch_time = elapsed / (batch_num + 1)
                est_remaining = avg_batch_time * remaining_batches
                progress_pct = ((batch_num + 1) / total_batches) * 100
                print(f" {progress_pct:.0f}% (ETA: {est_remaining/60:.0f}min)", end="", flush=True)
    
    print("] ✅")
    
    results["processing_time"] = time.time() - start_time
    results["files_per_hour"] = results["processed_files"] / (results["processing_time"] / 3600) if results["processing_time"] > 0 else 0
    
    return results

def filter_xml_files_for_tax_production(xml_files: List[Path], max_to_check: int = None) -> List[Path]:
    """Production filtering with minimal output"""
    if max_to_check:
        xml_files = xml_files[:max_to_check]
    
    tax_relevant_files = []
    
    for xml_file in xml_files:
        try:
            metadata, articles = parse_lovdata_xml(xml_file)
            if is_tax_relevant_document_silent(metadata, articles):
                tax_relevant_files.append(xml_file)
        except Exception:
            continue  # Skip files with errors silently
    
    return tax_relevant_files

def is_tax_relevant_document_silent(metadata: LawMetadata, articles: List[LawArticle]) -> bool:
    """Silent version for production"""
    # Same logic as original but no print statements
    tax_keywords = [
        'skatt', 'skatteloven', 'skatteetaten', 'skatteoppkrever', 
        'inntektsskatt', 'formuesskatt', 'selskapsskatt', 'eiendomsskatt',
        'merverdiavgift', 'mva', 'toll', 'tollavgift',
        'særavgift', 'arveavgift', 'dokumentavgift', 'stempelavgift',
        'skatteberegning', 'avgiftssats', 'avgiftsfri', 'skattefri', 
        'avgiftsplikt', 'skatteplikt',
    ]
    
    core_tax_departments = [
        'Finansdepartementet',
        'Nærings- og fiskeridepartementet',
    ]
    
    # Core department check
    if any(dept in metadata.department for dept in core_tax_departments):
        return True
    
    # Title keyword check
    title_lower = metadata.title.lower()
    if any(keyword in title_lower for keyword in tax_keywords):
        return True
    
    # Content keyword check (first 3 articles)
    for article in articles[:3]:
        content_lower = article.content.lower()
        if any(keyword in content_lower for keyword in tax_keywords):
            return True
    
    return False

def process_xml_files_semantically_silent(xml_files: List[Path], config: EmbeddingConfig) -> List[DocumentChunk]:
    """Silent processing for production"""
    all_chunks = []
    
    for xml_file in xml_files:
        try:
            metadata, articles = parse_lovdata_xml(xml_file)
            
            # Process articles (no size splitting for simplicity in production)
            for i, article in enumerate(articles):
                if article.content.strip():  # Only non-empty articles
                    chunk_metadata = {
                        'source_file': xml_file.name,
                        'created_at': datetime.now().isoformat(),
                        'estimated_tokens': article.token_count,
                        'document_id': metadata.document_id,
                        'document_title': metadata.title,
                        'article_number': article.article_number,
                        'article_title': article.title,
                        'section_title': article.section_title,
                        'hierarchy_path': article.hierarchy_path,
                        'chunk_type': article.chunk_type,
                        'legal_areas': metadata.legal_areas,
                        'department': metadata.department,
                        'last_modified': metadata.last_modified,
                        'ref_id': metadata.ref_id
                    }
                    
                    chunk = DocumentChunk(
                        text=f"{article.title}\n\n{article.content}",
                        metadata=chunk_metadata
                    )
                    all_chunks.append(chunk)
                    
        except Exception:
            continue  # Skip problematic files silently
    
    return all_chunks

def embed_chunks_silent(chunks: List[DocumentChunk]) -> List[DocumentChunk]:
    """Silent embedding for production"""
    for chunk in chunks:
        try:
            response = openai.embeddings.create(
                input=chunk.text,
                model=config.embedding_model
            )
            chunk.metadata['embedding'] = response.data[0].embedding
            chunk.metadata['embedding_model'] = config.embedding_model
        except Exception:
            continue  # Skip embedding errors silently
    return chunks

def ingest_to_elasticsearch_silent(chunks: List[DocumentChunk], index_name: str) -> int:
    """Silent ingest for production"""
    count = 0
    for chunk in chunks:
        if 'embedding' in chunk.metadata:
            try:
                doc = {
                    "content": chunk.text,
                    "document_id": chunk.metadata.get('document_id', ''),
                    "article_number": chunk.metadata.get('article_number', ''),
                    "embedding": chunk.metadata['embedding'],
                    # Add all metadata for rich search
                    "document_title": chunk.metadata.get('document_title', ''),
                    "article_title": chunk.metadata.get('article_title', ''),
                    "section_title": chunk.metadata.get('section_title', ''),
                    "department": chunk.metadata.get('department', ''),
                    "legal_areas": chunk.metadata.get('legal_areas', []),
                    "hierarchy_path": chunk.metadata.get('hierarchy_path', ''),
                    "chunk_type": chunk.metadata.get('chunk_type', ''),
                    "last_modified": chunk.metadata.get('last_modified', ''),
                    "estimated_tokens": chunk.metadata.get('estimated_tokens', 0)
                }
                
                es_client.index(index=index_name, body=doc)
                count += 1
            except Exception:
                continue  # Skip ingest errors silently
    return count


In [11]:
# Uncomment the line below to start production ingestion
production_results = run_production_ingestion()

🚀 STARTING PRODUCTION INGESTION
📊 Target index: lovdata_semantic_ada3l_251108
📂 Data directory: ../data/extracted
🔍 Filtering strategy: 2 core depts + 5 keyword-filtered + strict keywords
⚡ Processing in batches of 50
📊 Found 4222 XML files in dataset
🔍 Applying refined tax filtering...
✅ Filtering complete: 1193 tax-relevant files (71.7% reduction)
⚡ Processing 1193 files in 24 batches
📈 Progress: [██ 8% (ETA: 47min)██ 17% (ETA: 55min)██ 25% (ETA: 58min)██ 33% (ETA: 62min)██ 42% (ETA: 59min)██ 50% (ETA: 54min)██ 58% (ETA: 45min)██ 67% (ETA: 35min)██ 75% (ETA: 25min)██ 83% (ETA: 16min)██ 92% (ETA: 8min)██] ✅

🏁 PRODUCTION INGESTION COMPLETE
📄 Total XML files found: 4222
✅ Tax-relevant files processed: 1193
📉 Document reduction: 71.7%
📝 Total legal articles: 0
📦 Total chunks created: 10005
🤖 Chunks embedded: 10005
📇 Chunks ingested to ES: 826
⏱️ Total processing time: 1.8 hours
📈 Processing rate: 646.8 files/hour

✅ SUCCESS: Index 'lovdata_semantic_ada3l_251108' ready for production!
💡 

In [137]:
# 🔍 SIMPLE SEARCH TEST - Test the semantic chunks like your app does

def test_semantic_search(index_name: str):
    """
    Simple test that replicates your app's search exactly
    """
    # Test with a tax-related query
    query_text = "gebyr ved behandling"
    
    print(f"🔍 Testing search in index: {index_name}")
    print(f"🔎 Query: '{query_text}'")
    
    try:
        # Step 1: Embed the query (exactly like queryService.ts)
        embedding_response = openai.embeddings.create(
            input=query_text,
            model=config.embedding_model
        )
        search_vector = embedding_response.data[0].embedding
        
        # Step 2: Vector search (exactly like your app)
        es_response = es_client.search(
            index=index_name,
            size=5,  # Top 5 results
            knn={
                "field": "embedding",
                "query_vector": search_vector,
                "k": 20,
                "num_candidates": 100,
                "boost": 0.1
            }
        )
        
        hits = es_response['hits']['hits']
        print(f"✅ Found {len(hits)} results:")
        
        if hits:
            for i, hit in enumerate(hits, 1):
                source = hit['_source']
                score = hit['_score']
                doc_id = source.get('document_id', 'N/A')
                article = source.get('article_number', 'N/A')
                content = source.get('content', '')[:150]
                
                print(f"\n{i}. Score: {score:.3f}")
                print(f"   Doc: {doc_id}")
                print(f"   Article: {article}")
                print(f"   Content: {content}...")
                
            return True
        else:
            print("❌ No results found")
            return False
            
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return False

# Test the search on the chunks we just created
test_index = "lovdata_test_semantic"  # This is the test index created above
print(f"\n🔍 FINAL SEARCH TEST")
print("=" * 50)

# Check if test index has any documents
try:
    count = es_client.count(index=test_index)["count"]
    print(f"📊 Documents in {test_index}: {count}")
    
    if count > 0:
        search_success = test_semantic_search(test_index)
        if search_success:
            print(f"\n🎉 SEARCH TEST PASSED!")
            print(f"💡 The semantic chunks work with your app's search pattern")
            print(f"💡 Production index will be: {config.elasticsearch_index}")
        else:
            print(f"\n⚠️ Search test had issues - but the pipeline works")
    else:
        print(f"⚠️ No documents found in test index")
        
except Exception as e:
    print(f"❌ Could not check test index: {e}")

print(f"\n📋 INDEX SUMMARY:")
print(f"  🧪 Test index: lovdata_test_semantic (temporary)")
print(f"  🏭 Production index: {config.elasticsearch_index}")
print(f"  📚 Your old index: skatt_chunks_2025 (unchanged)")


🔍 FINAL SEARCH TEST
📊 Documents in lovdata_test_semantic: 132
🔍 Testing search in index: lovdata_test_semantic
🔎 Query: 'gebyr ved behandling'
✅ Found 5 results:

1. Score: 0.090
   Doc: FOR-2023-09-05-2500
   Article: § 4-3
   Content: Vilkår for å gå opp til sluttvurdering\n\nBetalt studie- og semesteravgift\n\nGjennomført og bestått praksis både i inn- og utlandet med mindre enn 10...

2. Score: 0.090
   Doc: FOR-2023-01-30-111
   Article: § 45
   Content: Saksbehandlingsregler\n\nKlager behandles i henhold til forvaltningslovens bestemmelser....

3. Score: 0.090
   Doc: FOR-2023-01-30-111
   Article: § 45
   Content: Saksbehandlingsregler\n\nKlager behandles i henhold til forvaltningslovens bestemmelser....

4. Score: 0.090
   Doc: FOR-2023-01-30-111
   Article: § 17b
   Content: Helseerklæring for utrykningskjøring\n\nStudenten må innhente og betale ny helseattest for utrykningskjøretøy etter forskrift 12. juni 2009 nr. 637 om...

5. Score: 0.090
   Doc: FOR-2023-01-30-111
   Art

## 5. Smart Processing Pipeline for Lovdata Dataset

In [124]:
def process_lovdata_dataset(data_dir: Path, config: EmbeddingConfig, max_files: int = None, use_tax_filter: bool = True) -> Dict[str, Any]:
    """
    Process entire Lovdata dataset with semantic chunking
    
    Args:
        data_dir: Path to extracted Lovdata data 
        config: EmbeddingConfig
        max_files: Limit for testing (None = all files)
        use_tax_filter: Filter for tax-relevant documents only
        
    Returns:
        Processing results
    """
    
    # Find all XML files
    xml_files = list(data_dir.glob("**/*.xml"))
    
    # Store original count before any filtering
    original_file_count = len(xml_files)
    
    if max_files:
        xml_files = xml_files[:max_files]
        print(f"🚧 Processing first {max_files} files for testing")
    
    print(f"📊 Found {len(xml_files)} XML files")
    
    # Apply tax filtering if enabled
    if use_tax_filter:
        xml_files = filter_xml_files_for_tax(xml_files, max_to_check=max_files)
        print(f"📊 After filtering: {len(xml_files)} tax-relevant files to process")
    else:
        print(f"📊 Processing all {len(xml_files)} files (no filtering)")
    
    results = {
        "processed_files": 0,
        "total_articles": 0,
        "total_chunks": 0,
        "total_embedded": 0,
        "total_ingested": 0,
        "oversized_articles": 0,
        "processing_time": 0,
        "files_before_filter": original_file_count if max_files is None else min(max_files, original_file_count),
        "files_after_filter": len(xml_files)
    }
    
    if not xml_files:
        print("❌ No files to process after filtering!")
        return results
        
    start_time = time.time()
    
    # Process in batches
    batch_size = config.batch_size
    total_batches = (len(xml_files) + batch_size - 1) // batch_size
    
    print(f"⚡ Processing {len(xml_files)} files in {total_batches} batches of {batch_size}")
    
    for batch_num in range(total_batches):
        batch_start = time.time()
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(xml_files))
        batch_files = xml_files[start_idx:end_idx]
        
        print(f"\\n🔄 Batch {batch_num + 1}/{total_batches}: Processing {len(batch_files)} files...")
        
        # Process semantic chunks
        batch_chunks = process_xml_files_semantically(batch_files, config)
        
        if batch_chunks:
            # Embed chunks
            print(f"🤖 Embedding {len(batch_chunks)} chunks...")
            embedded_chunks = embed_chunks(batch_chunks)
            
            # Ingest to Elasticsearch
            print(f"📇 Ingesting to ES index: {config.elasticsearch_index}")
            ingested_count = ingest_to_elasticsearch(embedded_chunks, config.elasticsearch_index)
            
            results["total_embedded"] += len(embedded_chunks)
            results["total_ingested"] += ingested_count
            results["total_chunks"] += len(batch_chunks)
        
        results["processed_files"] += len(batch_files)
        
        batch_time = time.time() - batch_start
        remaining_batches = total_batches - batch_num - 1
        est_remaining = batch_time * remaining_batches
        
        print(f"✅ Batch completed in {batch_time:.1f}s")
        if remaining_batches > 0:
            print(f"⏱️ Estimated remaining time: {est_remaining/60:.1f} minutes")
    
    results["processing_time"] = time.time() - start_time
    
    return results


def process_lovdata_dataset_filtered(data_dir: Path, config: EmbeddingConfig, max_files: int = None) -> Dict[str, Any]:
    """
    Convenience function to process only tax-relevant documents
    """
    return process_lovdata_dataset(data_dir, config, max_files, use_tax_filter=True)

In [125]:
  # Check the test index contents
  response = es_client.search(
      index="lovdata_test_semantic",
      size=10,
      body={"query": {"match_all": {}}}
  )

  print("📋 Sample documents in index:")
  for i, hit in enumerate(response['hits']['hits'][:5], 1):
      source = hit['_source']
      print(f"\n{i}. Document: {source.get('document_id', 'N/A')}")
      print(f"   Article: {source.get('article_number', 'N/A')}")
      print(f"   Content: {source.get('content', '')[:100]}...")

📋 Sample documents in index:

1. Document: FOR-2023-09-05-2500
   Article: § 1-1
   Content: Forskriftens formål og virkeområde\n\nDenne forskriften gjelder for fagskoleutdanningen Ettårsenhet ...

2. Document: FOR-2023-09-05-2500
   Article: § 2-1
   Content: Kvalifisering for opptak\n\nFagskolen Essens AS tar inn studenter som har fullført og bestått videre...

3. Document: FOR-2023-09-05-2500
   Article: § 2-2
   Content: Kvalifisering ved vurdering av realkompetanse\n\nFagskolen Essens AS kan ta inn studenter som er 23 ...

4. Document: FOR-2023-09-05-2500
   Article: § 2-3
   Content: Rangering ved hovedopptak\n\nDet er rektor i samråd med avdelingsleder som effektuerer opptak.\n\nDa...

5. Document: FOR-2023-09-05-2500
   Article: § 2-4
   Content: Fremvisning av politiattest ved opptak til fagskolen\n\nEtter opptak vil alle studenter ha krav om å...


## 6. Test Semantic Pipeline

In [126]:
# Test the complete semantic pipeline
print("🚀 Testing Lovdata Semantic Chunking Pipeline")
print(f"📊 Dataset contains: 4,222 XML files")
print(f"🎯 Target index: {config.elasticsearch_index}")

# Run test with 3 files
results = test_semantic_pipeline()

# Show comparison with current approach
print(f"\\n📈 SEMANTIC CHUNKING BENEFITS:")
print(f"✅ Semantic boundaries (complete legal articles)")
print(f"✅ Rich metadata for filtering and search")  
print(f"✅ Natural size variation vs fixed chunks")
print(f"✅ Legal context preservation")

print(f"\\n🗂️ Ready for full dataset processing!")
print(f"💡 To process all 4,222 files:")
print(f"   results = process_lovdata_dataset(Path('../data/extracted'), config)")
print(f"\\n⚡ Estimated processing time for full dataset: ~8-12 hours")

🚀 Testing Lovdata Semantic Chunking Pipeline
📊 Dataset contains: 4,222 XML files
🎯 Target index: lovdata_selected_semantic_251107
🧪 Testing semantic pipeline with 3 files...
🚧 Processing first 3 files for testing
📊 Found 3 XML files
🎯 Filtering 3 files for tax relevance...
📋 REFINED FILTERING STRATEGY (REDUCED):
   ✅ ALL docs from:
     🏦 Finansdepartementet
     🏢 Nærings- og fiskeridepartementet
   🔍 STRICT KEYWORD-FILTERED docs from:
     ⚖️ Justis- og beredskapsdepartementet
     👥 Arbeids- og sosialdepartementet
     🚗 Samferdselsdepartementet
     🌱 Klima- og miljødepartementet
     🌾 Landbruks- og matdepartementet
     📋 All other departments
   🚫 REMOVED DEPTS: Kommunal/inkludering (municipal focus)
   🚫 REMOVED KEYWORDS: 'gebyr', 'avgift' (too generic)

📊 REFINED FILTERING RESULTS:
  📄 Original files: 3
  ✅ Tax relevant total: 0
     🏦 From core departments: 0 (0.0%)
     🔍 From keyword filtering: 0 (0.0%)
  📉 Reduction: 100.0%
📊 After filtering: 0 tax-relevant files to proces

In [127]:
count = es_client.count(index="skatt_chunks_2025")["count"]
print(f"Chunks i ES: {count}")

Chunks i ES: 3045


In [ ]:
count = es_client.count(index="skatt_chunks_2025")["count"]
print(f"Chunks i ES: {count}")

## 7. Production Index Testing & Comparison

In [138]:
# 📊 INDEX ANALYSIS & COMPARISON TOOLS

def analyze_index_stats(index_name: str) -> Dict[str, Any]:
    """
    Get comprehensive statistics about an ES index
    """
    print(f"📊 ANALYZING INDEX: {index_name}")
    print("=" * 60)
    
    try:
        # Basic index info
        if not es_client.indices.exists(index=index_name):
            print(f"❌ Index '{index_name}' does not exist!")
            return {}
            
        # Document count
        count_response = es_client.count(index=index_name)
        total_docs = count_response["count"]
        
        # Index size and settings
        stats_response = es_client.indices.stats(index=index_name)
        index_stats = stats_response["indices"][index_name]
        
        # Sample documents for analysis
        sample_response = es_client.search(
            index=index_name,
            size=5,
            body={"query": {"match_all": {}}}
        )
        
        # Calculate token statistics from sample
        token_counts = []
        content_lengths = []
        departments = set()
        chunk_types = set()
        
        for hit in sample_response["hits"]["hits"]:
            source = hit["_source"]
            content = source.get("content", "")
            content_lengths.append(len(content))
            
            # Estimate tokens (4 chars per token for Norwegian)
            estimated_tokens = len(content) // 4
            token_counts.append(estimated_tokens)
            
            # Collect metadata
            departments.add(source.get("department", "Unknown"))
            chunk_types.add(source.get("chunk_type", "Unknown"))
        
        # Aggregate token statistics
        stats = {
            "total_documents": total_docs,
            "index_size_bytes": index_stats["total"]["store"]["size_in_bytes"],
            "index_size_mb": round(index_stats["total"]["store"]["size_in_bytes"] / (1024 * 1024), 2),
            "avg_content_length": round(sum(content_lengths) / len(content_lengths)) if content_lengths else 0,
            "avg_tokens_per_chunk": round(sum(token_counts) / len(token_counts)) if token_counts else 0,
            "min_tokens": min(token_counts) if token_counts else 0,
            "max_tokens": max(token_counts) if token_counts else 0,
            "departments_found": list(departments),
            "chunk_types_found": list(chunk_types),
            "sample_documents": len(sample_response["hits"]["hits"])
        }
        
        # Print formatted results
        print(f"📄 Total documents: {stats['total_documents']:,}")
        print(f"💾 Index size: {stats['index_size_mb']:,} MB")
        print(f"📏 Average chunk length: {stats['avg_content_length']:,} characters")
        print(f"🔤 Average tokens per chunk: {stats['avg_tokens_per_chunk']:,}")
        print(f"📊 Token range: {stats['min_tokens']} - {stats['max_tokens']}")
        print(f"🏢 Departments: {len(stats['departments_found'])}")
        print(f"📦 Chunk types: {stats['chunk_types_found']}")
        
        return stats
        
    except Exception as e:
        print(f"❌ Error analyzing index: {e}")
        return {}

def compare_index_sizes(new_index: str, old_index: str = "skatt_chunks_2025"):
    """
    Compare two indices for size, content, and efficiency
    """
    print(f"⚖️ COMPARING INDICES")
    print("=" * 60)
    print(f"🆕 New index: {new_index}")
    print(f"📚 Old index: {old_index}")
    print()
    
    new_stats = analyze_index_stats(new_index)
    print()
    old_stats = analyze_index_stats(old_index)
    
    if new_stats and old_stats:
        print(f"\n📊 COMPARISON SUMMARY")
        print("=" * 40)
        
        # Document count comparison
        doc_ratio = new_stats['total_documents'] / old_stats['total_documents']
        print(f"📄 Documents: {new_stats['total_documents']:,} vs {old_stats['total_documents']:,} ({doc_ratio:.2f}x)")
        
        # Size comparison
        size_ratio = new_stats['index_size_mb'] / old_stats['index_size_mb']
        print(f"💾 Size: {new_stats['index_size_mb']} MB vs {old_stats['index_size_mb']} MB ({size_ratio:.2f}x)")
        
        # Token comparison
        token_ratio = new_stats['avg_tokens_per_chunk'] / old_stats['avg_tokens_per_chunk'] if old_stats['avg_tokens_per_chunk'] > 0 else 0
        print(f"🔤 Avg tokens: {new_stats['avg_tokens_per_chunk']} vs {old_stats['avg_tokens_per_chunk']} ({token_ratio:.2f}x)")
        
        # Efficiency metrics
        docs_per_mb_new = new_stats['total_documents'] / new_stats['index_size_mb'] if new_stats['index_size_mb'] > 0 else 0
        docs_per_mb_old = old_stats['total_documents'] / old_stats['index_size_mb'] if old_stats['index_size_mb'] > 0 else 0
        
        print(f"⚡ Density: {docs_per_mb_new:.0f} vs {docs_per_mb_old:.0f} docs/MB")
        
        # Content quality assessment
        print(f"\n📋 CONTENT QUALITY")
        print("=" * 30)
        print(f"🆕 New: {new_stats['chunk_types_found']} (semantic articles)")
        print(f"📚 Old: {old_stats['chunk_types_found']} (fixed chunks)")
        
        if new_stats['avg_tokens_per_chunk'] > old_stats['avg_tokens_per_chunk']:
            print(f"✅ New index has {token_ratio:.1f}x larger chunks (better context)")
        else:
            print(f"⚠️ New index has smaller chunks")

# Run index analysis
print("🔍 ANALYZING PRODUCTION INDICES")
print("=" * 60)

# Analyze the new semantic index (if it exists)
new_index = config.elasticsearch_index
analyze_index_stats(new_index)

print(f"\n" + "="*60)

# Compare with old index
compare_index_sizes(new_index, "skatt_chunks_2025")

🔍 ANALYZING PRODUCTION INDICES
📊 ANALYZING INDEX: lovdata_selected_semantic_251107
📄 Total documents: 8,878
💾 Index size: 266.72 MB
📏 Average chunk length: 1,040 characters
🔤 Average tokens per chunk: 260
📊 Token range: 108 - 584
🏢 Departments: 1
📦 Chunk types: ['article']

⚖️ COMPARING INDICES
🆕 New index: lovdata_selected_semantic_251107
📚 Old index: skatt_chunks_2025

📊 ANALYZING INDEX: lovdata_selected_semantic_251107
📄 Total documents: 8,878
💾 Index size: 266.72 MB
📏 Average chunk length: 1,040 characters
🔤 Average tokens per chunk: 260
📊 Token range: 108 - 584
🏢 Departments: 1
📦 Chunk types: ['article']

📊 ANALYZING INDEX: skatt_chunks_2025
📄 Total documents: 3,045
💾 Index size: 96.32 MB
📏 Average chunk length: 3,108 characters
🔤 Average tokens per chunk: 776
📊 Token range: 763 - 790
🏢 Departments: 1
📦 Chunk types: ['Unknown']

📊 COMPARISON SUMMARY
📄 Documents: 8,878 vs 3,045 (2.92x)
💾 Size: 266.72 MB vs 96.32 MB (2.77x)
🔤 Avg tokens: 260 vs 776 (0.34x)
⚡ Density: 33 vs 32 docs/M

In [139]:
# 🔍 HYBRID SEARCH TESTING - Test exactly like your app does

def test_hybrid_search_comparison(query_text: str, new_index: str, old_index: str = "skatt_chunks_2025"):
    """
    Test the same query on both indices using your app's exact hybrid search pattern
    """
    print(f"🔍 HYBRID SEARCH COMPARISON")
    print("=" * 60)
    print(f"🔎 Query: '{query_text}'")
    print(f"🆕 New index: {new_index}")
    print(f"📚 Old index: {old_index}")
    
    # Step 1: Embed the query (same for both)
    try:
        embedding_response = openai.embeddings.create(
            input=query_text,
            model=config.embedding_model
        )
        search_vector = embedding_response.data[0].embedding
        print(f"✅ Query embedded ({len(search_vector)} dimensions)")
    except Exception as e:
        print(f"❌ Embedding failed: {e}")
        return
    
    # Test both indices
    results = {}
    
    for idx_name, idx_label in [(new_index, "🆕 NEW"), (old_index, "📚 OLD")]:
        print(f"\n{idx_label} INDEX: {idx_name}")
        print("-" * 40)
        
        try:
            # Your app's exact hybrid search pattern from esSearchConsumer.ts
            es_response = es_client.search(
                index=idx_name,
                size=5,  # ES_SEARCH_NUM_HITS from your app
                knn={
                    "field": "embedding",
                    "query_vector": search_vector,
                    "k": 20,  # ES_KNN_NUMBER from your app
                    "num_candidates": 100,
                    "boost": 1.0
                }
            )
            
            hits = es_response['hits']['hits']
            results[idx_name] = hits
            
            print(f"✅ Found {len(hits)} results")
            
            # Analyze results
            if hits:
                scores = [hit['_score'] for hit in hits]
                content_lengths = []
                token_counts = []
                
                print(f"📊 Score range: {min(scores):.3f} - {max(scores):.3f}")
                
                for i, hit in enumerate(hits, 1):
                    source = hit['_source']
                    content = source.get('content', '')
                    content_lengths.append(len(content))
                    token_counts.append(len(content) // 4)  # Estimate tokens
                    
                    print(f"  {i}. Score: {hit['_score']:.3f}")
                    print(f"     Doc: {source.get('document_id', 'N/A')}")
                    print(f"     Length: {len(content)} chars (~{len(content)//4} tokens)")
                    print(f"     Preview: {content[:100]}...")
                    print()
                
                avg_length = sum(content_lengths) / len(content_lengths)
                avg_tokens = sum(token_counts) / len(token_counts)
                total_context_chars = sum(content_lengths)
                total_context_tokens = sum(token_counts)
                
                print(f"📏 Average chunk: {avg_length:.0f} chars ({avg_tokens:.0f} tokens)")
                print(f"📦 Total context: {total_context_chars:,} chars ({total_context_tokens:,} tokens)")
                
            else:
                print("❌ No results found")
                
        except Exception as e:
            print(f"❌ Search failed: {e}")
    
    # Compare results
    if len(results) == 2:
        new_hits = results.get(new_index, [])
        old_hits = results.get(old_index, [])
        
        if new_hits and old_hits:
            print(f"\n⚖️ CONTEXT SIZE COMPARISON")
            print("=" * 40)
            
            # Calculate total context for each
            new_total_chars = sum(len(hit['_source'].get('content', '')) for hit in new_hits)
            old_total_chars = sum(len(hit['_source'].get('content', '')) for hit in old_hits)
            
            new_total_tokens = new_total_chars // 4
            old_total_tokens = old_total_chars // 4
            
            print(f"🆕 New index context: {new_total_chars:,} chars ({new_total_tokens:,} tokens)")
            print(f"📚 Old index context: {old_total_chars:,} chars ({old_total_tokens:,} tokens)")
            
            if new_total_tokens > old_total_tokens:
                ratio = new_total_tokens / old_total_tokens
                print(f"✅ New index provides {ratio:.1f}x more context!")
            else:
                ratio = old_total_tokens / new_total_tokens
                print(f"⚠️ Old index provides {ratio:.1f}x more context")
                
            # Analyze chunk quality
            new_avg_tokens = new_total_tokens / len(new_hits)
            old_avg_tokens = old_total_tokens / len(old_hits)
            
            print(f"\n📊 CHUNK QUALITY")
            print(f"🆕 New avg chunk: {new_avg_tokens:.0f} tokens (semantic articles)")
            print(f"📚 Old avg chunk: {old_avg_tokens:.0f} tokens (fixed chunks)")

# Test with tax-related queries
test_queries = [
    "merverdiavgift på tjenester",
    "fradrag for reiseutgifter", 
    "selskapsskatt beregning",
    "arveavgift og gave",
    "eiendomsskatt verdsetting"
]

print("🧪 RUNNING HYBRID SEARCH TESTS")
print("=" * 60)

for i, query in enumerate(test_queries, 1):
    print(f"\n🔍 TEST {i}/{len(test_queries)}")
    test_hybrid_search_comparison(query, config.elasticsearch_index)
    if i < len(test_queries):
        print("\n" + "="*80 + "\n")

🧪 RUNNING HYBRID SEARCH TESTS

🔍 TEST 1/5
🔍 HYBRID SEARCH COMPARISON
🔎 Query: 'merverdiavgift på tjenester'
🆕 New index: lovdata_selected_semantic_251107
📚 Old index: skatt_chunks_2025
✅ Query embedded (1536 dimensions)

🆕 NEW INDEX: lovdata_selected_semantic_251107
----------------------------------------
✅ Found 5 results
📊 Score range: 0.931 - 0.936
  1. Score: 0.936
     Doc: FOR-2012-11-14-1066
     Length: 101 chars (~25 tokens)
     Preview: Merverdiavgift mv.

Alle priser det opplyses om skal inkludere merverdiavgift og offentlige avgifter...

  2. Score: 0.934
     Doc: LOV-2009-06-19-58
     Length: 195 chars (~48 tokens)
     Preview: Persontransport mv.

Det skal beregnes merverdiavgift med redusert sats ved omsetning og uttak av tj...

  3. Score: 0.932
     Doc: LOV-2009-06-19-58
     Length: 182 chars (~45 tokens)
     Preview: Fornøyelsesparker mv.

Det skal beregnes merverdiavgift med redusert sats ved omsetning, uttak og fo...

  4. Score: 0.931
     Doc: FOR-2009-12-

In [140]:
# 🔬 SAMPLE DATA & METADATA INSPECTION

def inspect_sample_documents(index_name: str, num_samples: int = 3):
    """
    Show detailed sample documents with full metadata and vector info
    """
    print(f"🔬 SAMPLE DOCUMENT INSPECTION: {index_name}")
    print("=" * 60)
    
    try:
        if not es_client.indices.exists(index=index_name):
            print(f"❌ Index '{index_name}' does not exist!")
            return
            
        # Get sample documents
        response = es_client.search(
            index=index_name,
            size=num_samples,
            body={"query": {"match_all": {}}}
        )
        
        hits = response["hits"]["hits"]
        
        if not hits:
            print("❌ No documents found in index")
            return
            
        for i, hit in enumerate(hits, 1):
            source = hit["_source"]
            
            print(f"\n📄 SAMPLE DOCUMENT {i}")
            print("-" * 40)
            
            # Basic info
            print(f"🆔 Document ID: {source.get('document_id', 'N/A')}")
            print(f"📰 Document Title: {source.get('document_title', 'N/A')}")
            print(f"📖 Article: {source.get('article_number', 'N/A')}")
            print(f"📝 Article Title: {source.get('article_title', 'N/A')}")
            print(f"🏢 Department: {source.get('department', 'N/A')}")
            
            # Content analysis
            content = source.get('content', '')
            content_length = len(content)
            estimated_tokens = content_length // 4
            
            print(f"📏 Content Length: {content_length:,} characters")
            print(f"🔤 Estimated Tokens: {estimated_tokens:,}")
            
            # Metadata richness
            metadata_fields = [
                'section_title', 'hierarchy_path', 'chunk_type', 
                'legal_areas', 'last_modified', 'estimated_tokens'
            ]
            
            print(f"🗂️ Rich Metadata:")
            for field in metadata_fields:
                value = source.get(field, 'N/A')
                if isinstance(value, list):
                    value = ', '.join(value) if value else 'None'
                print(f"   {field}: {value}")
            
            # Vector info
            embedding = source.get('embedding', [])
            if embedding:
                print(f"🧮 Vector: {len(embedding)} dimensions")
                print(f"   Sample values: [{', '.join(f'{v:.4f}' for v in embedding[:5])}...]")
            else:
                print(f"❌ No embedding found")
                
            # Content preview
            print(f"\n📖 Content Preview:")
            print(f"{'─' * 50}")
            print(content[:300] + "..." if len(content) > 300 else content)
            print(f"{'─' * 50}")
            
    except Exception as e:
        print(f"❌ Error inspecting documents: {e}")

def get_index_mapping(index_name: str):
    """
    Show the Elasticsearch mapping (schema) for the index
    """
    print(f"🗺️ INDEX MAPPING: {index_name}")
    print("=" * 60)
    
    try:
        if not es_client.indices.exists(index=index_name):
            print(f"❌ Index '{index_name}' does not exist!")
            return
            
        mapping = es_client.indices.get_mapping(index=index_name)
        properties = mapping[index_name]["mappings"].get("properties", {})
        
        print(f"📊 Total fields: {len(properties)}")
        print(f"\n📋 Field Types:")
        
        for field_name, field_config in properties.items():
            field_type = field_config.get("type", "unknown")
            
            if field_type == "dense_vector":
                dims = field_config.get("dims", "unknown")
                print(f"   🧮 {field_name}: {field_type} ({dims} dimensions)")
            elif field_type == "text":
                print(f"   📝 {field_name}: {field_type}")
            elif field_type == "keyword":
                print(f"   🔑 {field_name}: {field_type}")
            elif field_type == "date":
                print(f"   📅 {field_name}: {field_type}")
            elif field_type == "integer":
                print(f"   🔢 {field_name}: {field_type}")
            else:
                print(f"   ❓ {field_name}: {field_type}")
                
    except Exception as e:
        print(f"❌ Error getting mapping: {e}")

# Run inspections
print("🔬 DETAILED INDEX INSPECTION")
print("=" * 80)

# Inspect new semantic index
new_index = config.elasticsearch_index
print(f"🆕 NEW SEMANTIC INDEX")
get_index_mapping(new_index)
print()
inspect_sample_documents(new_index, 2)

print(f"\n" + "="*80)

# Compare with old index  
print(f"📚 OLD FIXED-CHUNK INDEX")
old_index = "skatt_chunks_2025"
get_index_mapping(old_index)
print()
inspect_sample_documents(old_index, 2)

🔬 DETAILED INDEX INSPECTION
🆕 NEW SEMANTIC INDEX
🗺️ INDEX MAPPING: lovdata_selected_semantic_251107
📊 Total fields: 13

📋 Field Types:
   📝 article_number: text
   📝 article_title: text
   📝 chunk_type: text
   📝 content: text
   📝 department: text
   📝 document_id: text
   📝 document_title: text
   🧮 embedding: dense_vector (1536 dimensions)
   ❓ estimated_tokens: long
   📝 hierarchy_path: text
   📅 last_modified: date
   📝 legal_areas: text
   📝 section_title: text

🔬 SAMPLE DOCUMENT INSPECTION: lovdata_selected_semantic_251107

📄 SAMPLE DOCUMENT 1
----------------------------------------
🆔 Document ID: FOR-2008-09-22-1080
📰 Document Title: Forskrift om risikostyring og internkontroll
📖 Article: § 2
📝 Article Title: Forholdsmessighet og virksomhet i enkeltpersonforetak
🏢 Department: Finansdepartementet
📏 Content Length: 432 characters
🔤 Estimated Tokens: 108
🗂️ Rich Metadata:
   section_title: Kapittel 1. Innledende bestemmelser
   hierarchy_path: FOR-2008-09-22-1080/§ 2
   chunk_typ

## 8. Content Verification - PDF vs Lovdata XML Coverage

In [ ]:
# 🔍 CONTENT COVERAGE VERIFICATION

def verify_pdf_content_coverage(old_index: str = "skatt_chunks_2025", new_index: str = None):
    """
    Verify that the content from original PDFs is covered in the new Lovdata semantic chunks
    """
    if new_index is None:
        new_index = config.elasticsearch_index
        
    print("🔍 CONTENT COVERAGE VERIFICATION")
    print("=" * 60)
    print(f"📚 Old PDF index: {old_index}")
    print(f"🆕 New Lovdata index: {new_index}")
    
    # Sample key tax terms that should be present in both
    key_tax_terms = [
        "inntektsskatt",
        "merverdiavgift", 
        "selskapsskatt",
        "fradrag",
        "skatteplikt",
        "arveavgift",
        "eiendomsskatt",
        "skatteberegning",
        "skatteetaten",
        "avgiftssats"
    ]
    
    print(f"\n🎯 Testing coverage for {len(key_tax_terms)} key tax terms...")
    
    coverage_results = {}
    
    for term in key_tax_terms:
        print(f"\n🔎 Testing term: '{term}'")
        print("-" * 30)
        
        # Search in old PDF index
        old_results = search_term_in_index(term, old_index)
        old_count = len(old_results) if old_results else 0
        
        # Search in new Lovdata index
        new_results = search_term_in_index(term, new_index)
        new_count = len(new_results) if new_results else 0
        
        coverage_results[term] = {
            "old_count": old_count,
            "new_count": new_count,
            "covered": new_count > 0,
            "improvement": new_count - old_count if old_count > 0 else "N/A"
        }
        
        if new_count > 0 and old_count > 0:
            ratio = new_count / old_count
            print(f"  📊 Old: {old_count} results, New: {new_count} results ({ratio:.1f}x)")
            if new_count >= old_count:
                print(f"  ✅ Well covered (equal or more results)")
            else:
                print(f"  ⚠️ Less coverage ({ratio:.1f}x fewer results)")
        elif new_count > 0:
            print(f"  ✅ New coverage: {new_count} results (not in old)")
        elif old_count > 0:
            print(f"  ❌ Missing coverage: {old_count} results in old, 0 in new")
        else:
            print(f"  ⚪ No results in either index")
    
    # Summary
    print(f"\n📊 COVERAGE SUMMARY")
    print("=" * 40)
    
    covered_terms = sum(1 for r in coverage_results.values() if r["covered"])
    total_terms = len(key_tax_terms)
    coverage_pct = (covered_terms / total_terms) * 100
    
    print(f"✅ Terms with coverage: {covered_terms}/{total_terms} ({coverage_pct:.1f}%)")
    
    missing_terms = [term for term, r in coverage_results.items() if not r["covered"]]
    if missing_terms:
        print(f"❌ Missing terms: {', '.join(missing_terms)}")
    
    improved_terms = [term for term, r in coverage_results.items() 
                     if isinstance(r["improvement"], int) and r["improvement"] > 0]
    if improved_terms:
        print(f"📈 Improved coverage: {', '.join(improved_terms)}")
        
    return coverage_results

def search_term_in_index(term: str, index_name: str, size: int = 10):
    """
    Search for a term in an ES index using text search
    """
    try:
        if not es_client.indices.exists(index=index_name):
            print(f"    ❌ Index '{index_name}' does not exist")
            return []
            
        response = es_client.search(
            index=index_name,
            size=size,
            body={
                "query": {
                    "match": {
                        "content": {
                            "query": term,
                            "minimum_should_match": "100%"
                        }
                    }
                }
            }
        )
        
        hits = response["hits"]["hits"]
        hit_count = len(hits)
        total_hits = response["hits"]["total"]["value"]
        
        print(f"    📊 Found {hit_count}/{total_hits} results")
        
        return hits
        
    except Exception as e:
        print(f"    ❌ Search failed: {e}")
        return []

# Run content coverage verification
verify_pdf_content_coverage()

In [ ]:
# 📋 DETAILED CONTENT COMPARISON

def compare_specific_tax_concepts(old_index: str = "skatt_chunks_2025", new_index: str = None):
    """
    Deep dive comparison of specific tax concepts between PDF and Lovdata sources
    """
    if new_index is None:
        new_index = config.elasticsearch_index
        
    print("📋 DETAILED TAX CONCEPT COMPARISON")
    print("=" * 60)
    
    # More specific tax concepts to test
    tax_concepts = {
        "§ 5-1": "Skatteloven § 5-1 (skatteplikt)",
        "§ 6-1": "Skatteloven § 6-1 (alminnelig inntekt)", 
        "§ 6-10": "Skatteloven § 6-10 (fradrag)",
        "merverdiavgiftsloven": "Merverdiavgiftsloven",
        "skattefradrag reise": "Fradrag for reiseutgifter",
        "selskapsbeskatning": "Selskapsskatt regler",
        "eiendomsskatt kommune": "Kommunal eiendomsskatt",
        "arveavgift satser": "Arveavgift satser og regler"
    }
    
    comparison_results = {}
    
    for concept_key, concept_desc in tax_concepts.items():
        print(f"\n🎯 Testing: {concept_desc}")
        print("─" * 50)
        
        # Search both indices
        old_hits = search_term_in_index(concept_key, old_index, size=3)
        new_hits = search_term_in_index(concept_key, new_index, size=3)
        
        comparison_results[concept_key] = {
            "description": concept_desc,
            "old_results": len(old_hits) if old_hits else 0,
            "new_results": len(new_hits) if new_hits else 0
        }
        
        # Show sample content from both
        if old_hits and len(old_hits) > 0:
            print(f"\n📚 Sample from OLD PDF index:")
            old_content = old_hits[0]["_source"]["content"][:200]
            print(f"   {old_content}...")
        else:
            print(f"\n📚 OLD PDF index: No results")
            
        if new_hits and len(new_hits) > 0:
            print(f"\n🆕 Sample from NEW Lovdata index:")
            new_content = new_hits[0]["_source"]["content"][:200]
            new_doc = new_hits[0]["_source"].get("document_id", "N/A")
            new_article = new_hits[0]["_source"].get("article_number", "N/A")
            print(f"   Doc: {new_doc}, Article: {new_article}")
            print(f"   {new_content}...")
        else:
            print(f"\n🆕 NEW Lovdata index: No results")
    
    # Analysis summary
    print(f"\n📊 DETAILED COMPARISON SUMMARY")
    print("=" * 50)
    
    fully_covered = 0
    partially_covered = 0
    missing = 0
    new_only = 0
    
    for concept_key, results in comparison_results.items():
        old_count = results["old_results"]
        new_count = results["new_results"]
        
        if old_count > 0 and new_count > 0:
            if new_count >= old_count:
                fully_covered += 1
                status = "✅ Fully covered"
            else:
                partially_covered += 1
                status = "⚠️ Partially covered"
        elif old_count > 0 and new_count == 0:
            missing += 1
            status = "❌ Missing"
        elif old_count == 0 and new_count > 0:
            new_only += 1
            status = "🆕 New content"
        else:
            status = "⚪ Not found in either"
            
        print(f"{status}: {results['description']}")
    
    print(f"\n📈 COVERAGE ANALYSIS:")
    print(f"✅ Fully covered: {fully_covered}")
    print(f"⚠️ Partially covered: {partially_covered}")
    print(f"❌ Missing from new: {missing}")
    print(f"🆕 New content only: {new_only}")
    
    total_testable = fully_covered + partially_covered + missing
    if total_testable > 0:
        coverage_rate = (fully_covered + partially_covered) / total_testable * 100
        print(f"📊 Overall coverage rate: {coverage_rate:.1f}%")
    
    return comparison_results

# Run detailed comparison
comparison_results = compare_specific_tax_concepts()

In [ ]:
# 🔬 DOCUMENT SOURCE ANALYSIS

def analyze_document_sources(new_index: str = None):
    """
    Analyze what types of legal documents are in the new Lovdata index
    """
    if new_index is None:
        new_index = config.elasticsearch_index
        
    print("🔬 DOCUMENT SOURCE ANALYSIS")
    print("=" * 60)
    print(f"📊 Analyzing sources in: {new_index}")
    
    try:
        if not es_client.indices.exists(index=new_index):
            print(f"❌ Index '{new_index}' does not exist!")
            return {}
            
        # Get sample documents to analyze document types
        response = es_client.search(
            index=new_index,
            size=100,  # Larger sample
            body={
                "query": {"match_all": {}},
                "_source": ["document_id", "document_title", "department", "legal_areas"]
            }
        )
        
        documents = {}
        departments = {}
        legal_areas = {}
        document_types = {"LOV": 0, "FOR": 0, "Other": 0}
        
        for hit in response["hits"]["hits"]:
            source = hit["_source"]
            doc_id = source.get("document_id", "")
            doc_title = source.get("document_title", "")
            dept = source.get("department", "Unknown")
            areas = source.get("legal_areas", [])
            
            # Count unique documents
            if doc_id not in documents:
                documents[doc_id] = {
                    "title": doc_title,
                    "department": dept,
                    "legal_areas": areas
                }
                
                # Categorize by document type (LOV/FOR prefix)
                if doc_id.startswith("LOV-"):
                    document_types["LOV"] += 1
                elif doc_id.startswith("FOR-"):
                    document_types["FOR"] += 1
                else:
                    document_types["Other"] += 1
            
            # Count departments
            departments[dept] = departments.get(dept, 0) + 1
            
            # Count legal areas
            for area in areas:
                legal_areas[area] = legal_areas.get(area, 0) + 1
        
        print(f"\n📊 DOCUMENT OVERVIEW")
        print("=" * 40)
        print(f"📄 Unique documents sampled: {len(documents)}")
        print(f"🏛️ Laws (LOV): {document_types['LOV']}")
        print(f"📋 Regulations (FOR): {document_types['FOR']}")
        print(f"❓ Other: {document_types['Other']}")
        
        print(f"\n🏢 TOP DEPARTMENTS")
        print("=" * 30)
        sorted_departments = sorted(departments.items(), key=lambda x: x[1], reverse=True)
        for dept, count in sorted_departments[:8]:
            print(f"  {count:3d} chunks: {dept}")
        
        print(f"\n⚖️ TOP LEGAL AREAS") 
        print("=" * 30)
        sorted_areas = sorted(legal_areas.items(), key=lambda x: x[1], reverse=True)
        for area, count in sorted_areas[:8]:
            print(f"  {count:3d} chunks: {area}")
            
        # Check for key tax-related documents
        print(f"\n🎯 KEY TAX DOCUMENTS CHECK")
        print("=" * 40)
        
        key_tax_docs = {
            "skatteloven": ["LOV-1999-03-26-14", "skatteloven"],
            "merverdiavgiftsloven": ["LOV-2009-06-19-58", "merverdiavgiftsloven"], 
            "arveavgiftsloven": ["LOV-2008-12-19-112", "arveavgiftsloven"],
            "eiendomsskatteloven": ["LOV-1975-06-06-29", "eiendomsskatteloven"]
        }
        
        found_docs = {}
        for doc_name, (doc_id_pattern, search_term) in key_tax_docs.items():
            # Check if document ID exists
            matching_docs = [doc_id for doc_id in documents.keys() if doc_id_pattern in doc_id or search_term.lower() in documents[doc_id]["title"].lower()]
            
            if matching_docs:
                found_docs[doc_name] = matching_docs[0]
                print(f"  ✅ {doc_name}: Found ({matching_docs[0]})")
            else:
                print(f"  ❌ {doc_name}: Not found in sample")
        
        return {
            "documents": documents,
            "document_types": document_types,
            "departments": departments, 
            "legal_areas": legal_areas,
            "key_tax_docs": found_docs
        }
        
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        return {}

def check_lovdata_vs_pdf_content_types():
    """
    High-level comparison of content types between PDF and Lovdata sources
    """
    print("\n🔄 CONTENT TYPE COMPARISON")
    print("=" * 60)
    print("📚 Original PDF sources likely included:")
    print("  - Skatteloven (LOV-1999-03-26-14)")
    print("  - Merverdiavgiftsloven (LOV-2009-06-19-58)")
    print("  - Various tax regulations and circulars")
    print("  - Tax administration guidelines")
    
    print("\n🆕 New Lovdata XML sources include:")
    print("  - All current Norwegian laws (LOV)")
    print("  - All current regulations (FOR)")
    print("  - Updated and consolidated legal text")
    print("  - Official department sources")
    
    print("\n💡 EXPECTED BENEFITS:")
    print("  ✅ More comprehensive coverage")
    print("  ✅ Always up-to-date legal text")
    print("  ✅ Broader legal context beyond core tax laws")
    print("  ✅ Semantic article boundaries (vs arbitrary PDF chunks)")
    print("  ✅ Rich metadata for better search relevance")

# Run document source analysis
doc_analysis = analyze_document_sources()
check_lovdata_vs_pdf_content_types()